# Entrenamiento y evaluación en Colab

Notebook principal nuevo del proyecto.

In [ ]:
# =========================
# 1. Setup Colab
# =========================
from google.colab import drive
drive.mount("/content/drive")

!git clone https://github.com/sebastianquispearias/tesis-seg.git
%cd tesis-seg
!pip install -q -r requirements.txt

import sys
sys.path.append("/content/tesis-seg")

# =========================
# 2. Imports
# =========================
from src.defaults import get_default_config, summarize_config
from src.augmentations import (
    get_supervised_train_augmentation,
    get_weak_augmentation,
    get_strong_augmentation,
)
from src.datasets import (
    build_supervised_datasets,
    build_unlabeled_datasets,
    build_dataloaders,
)
from src.train import run_training
from src.evaluate import evaluate_checkpoint
from src.visualization import show_config_summary, show_dataset_examples
from src.sanity_checks import run_boundary_sanity_checks

# =========================
# 3. Config
# =========================
cfg = get_default_config()

# RUTAS: en el notebook original IMG_ROOT = MSK_ROOT = DATA_ROOT
cfg["img_root"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"
cfg["exp_dir"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/runs/3a_unet_semi_no_temp_CLAHE_CORREGIDO"

cfg["arch"] = "unetpp"
cfg["backbone"] = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"] = 0
cfg["use_semi"] = True
cfg["use_temp_consistency"] = True  # lo mantenemos, pero sin efecto al poner lambda_t=0
cfg["image_preproc"] = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False

cfg["lambda_u"] = 0.05
cfg["tau"] = 0.95
cfg["ema_decay"] = 0.99
cfg["semi_start_epoch"] = 30
cfg["semi_warmup_epochs"] = 20

cfg["lambda_t"] = 0.00
cfg["max_temp_delta"] = 2
cfg["temp_start_epoch"] = 4000
cfg["temp_warmup_epochs"] = 5
cfg["tau_temp"] = 0.7

cfg["target_size"] = (320, 320)
cfg["use_pad"] = True
cfg["imagenet_norm"] = False

cfg["batch_size"] = 5
cfg["num_workers"] = 4
cfg["drop_last"] = True
cfg["num_augmented"] = 5

cfg["lr"] = 1e-3
cfg["weight_decay"] = 1e-4
cfg["epochs"] = 2000
cfg["warmup_epochs"] = 10
cfg["patience_es"] = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"
cfg["run_ruler_eval"] = True

print(summarize_config(cfg))
# =========================
# 4. Transforms / datasets
# =========================
train_tf = get_supervised_train_augmentation(cfg)
weak_tf = get_weak_augmentation(cfg)
strong_tf = get_strong_augmentation(cfg)

train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(cfg, weak_tf=weak_tf, strong_tf=strong_tf)

show_dataset_examples(val_ds, n=3)

loaders = build_dataloaders(
    cfg,
    train_ds=train_ds,
    val_ds=val_ds,
    test_ds=test_ds,
    unlabeled_ds=unlabeled_ds,
    temporal_unlab_ds=temporal_unlab_ds,
)

# =========================
# 5. Sanity checks
# =========================
run_boundary_sanity_checks(
    train_masks_dir=f"{cfg['msk_root']}/train/masks",
    tol_px=cfg["boundary_tol_px"],
)

# =========================
# 6. Train
# =========================
artifacts = run_training(cfg, loaders)

# =========================
# 7. Evaluate
# =========================
results = evaluate_checkpoint(
    cfg,
    artifacts["model"],
    loaders,
    artifacts["best_path"],
    artifacts["history"],
)
print(results)